# Task 1 — raising macro F1 with physics features

Branch `task1`. Goal: beat the WiFo2 linear-probe baseline on LoS/NLoS.

The baseline head scores **0.428** macro F1 (seed-averaged), barely above the
**0.375** majority-class floor. This notebook adds classical channel features
and reaches **0.880**.

Everything is seed-averaged over 10 seeds, because at 10 samples a single-seed
result is noise: sigma is 0.06-0.11, so any change below ~0.1 is unmeasurable.

In [ ]:
import sys, warnings, json
from pathlib import Path
from types import SimpleNamespace

warnings.filterwarnings('ignore')
ROOT = Path.cwd() if (Path.cwd() / 'model.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np, torch, scipy.io as sio, h5py
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import f1_score, confusion_matrix

from model import WiFo2_model
from main import freeze_wifo
from physics_features import extract, STABLE

SEEDS = [0, 1, 7, 13, 42, 99, 123, 2024, 31337, 555]
device = torch.device('cpu')

## 1. The noise floor

Before optimising anything, establish what counts as a real change. Same CV
protocol, ten different seeds.

| variant | mean | std |
|---|---|---|
| WiFo2 linear probe (the baseline head) | 0.428 | 0.090 |
| WiFo2 linear probe + weight decay | 0.395 | 0.114 |
| WiFo2 mean-pool probe | 0.561 | 0.060 |
| majority class | 0.375 | — |

Two things this settles:

- **mean-pooling is a real gain** (+0.13, wins on 9 of 10 seeds)
- **weight decay is not.** An earlier single-seed run showed 0.375 -> 0.524 and
  that was noise; averaged, it makes things slightly worse

Everything below is reported the same way.

In [ ]:
def load_mat(path, key):
    try:
        return sio.loadmat(path)[key]
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            a = f[key][...]
        if a.dtype.names == ('real', 'imag'):
            a = a['real'] + 1j * a['imag']
        return a.transpose(range(a.ndim)[::-1])

H   = load_mat(ROOT / 'dataset/Task1/H_train.mat', 'H')
y   = load_mat(ROOT / 'dataset/Task1/L_train.mat', 'label').ravel().astype(int)
H_te = load_mat(ROOT / 'dataset/Task1/H_test.mat', 'H')

print(f'train {H.shape}  labels {y}  balance {np.bincount(y)}')
print(f'test  {H_te.shape}  (unlabelled)')

## 2. Physics features

A direct path leaves signatures a 10-sample-trained probe cannot learn but that
40 years of channel modelling already names:

| feature | why it separates LoS from NLoS |
|---|---|
| `first_tap_frac` | fraction of power in delay tap 0. A direct path puts most energy there |
| `rms_delay` | RMS delay spread. NLoS spreads energy over many reflections |
| `temporal_corr` | LoS channels are dominated by a stable path, so they decorrelate slowly |
| `K_max`, `K_mean` | Rician K-factor: direct power over scattered power |
| `cv`, `kurt`, `skew` | amplitude distribution shape. Rayleigh (NLoS) has CV = 0.523 |
| `coh_bw`, `freq_corr_mean` | coherence bandwidth. Short delay spread => wide coherence |

Implemented in `physics_features.py`, which carries a synthetic self-check
(`python physics_features.py`) verifying that a simulated LoS channel scores
higher `first_tap_frac`, lower `rms_delay` and higher `K_max` than an NLoS one.

In [ ]:
X_phys, names = extract(H)
X_phys_te, _  = extract(H_te)
print(f'{len(names)} features, train {X_phys.shape}, test {X_phys_te.shape}\n')

print(f"{'feature':18s}{'class 0 mean':>14s}{'class 1 mean':>14s}{'t-stat':>9s}")
tstat = {}
for j, n in enumerate(names):
    a, b = X_phys[y == 0, j], X_phys[y == 1, j]
    t = abs(a.mean() - b.mean()) / (np.sqrt(a.var()/len(a) + b.var()/len(b)) + 1e-12)
    tstat[n] = t
    print(f'{n:18s}{a.mean():>14.4f}{b.mean():>14.4f}{t:>9.2f}' + ('   <- stable' if n in STABLE else ''))

### What the numbers say about the labels

`first_tap_frac` is 0.028 for class 0 and 0.491 for class 1. Class 1 puts half
its energy in the first delay tap and has the shorter delay spread — that is a
direct path.

**Class 1 = LoS, class 0 = NLoS.** The repo never states the mapping; the
physics does.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, n in zip(axes, STABLE):
    j = names.index(n)
    for cls, colour in [(0, 'tab:blue'), (1, 'tab:orange')]:
        v = X_phys[y == cls, j]
        ax.scatter(v, np.full_like(v, cls) + np.random.uniform(-.06, .06, len(v)),
                   c=colour, s=70, label=f'class {cls}' if ax is axes[0] else None)
    ax.set(title=f'{n}  (t={tstat[n]:.2f})', yticks=[0, 1],
           yticklabels=['0 = NLoS', '1 = LoS'])
    ax.grid(alpha=.3)
axes[0].legend()
plt.tight_layout(); plt.show()

## 3. WiFo2 features, for comparison

Mean-pooled encoder output, 64 dims. Same extraction as
`notebook/task1_train.ipynb` section 3.

In [ ]:
def wifo_features(Hc):
    x = torch.stack([torch.tensor(Hc.real), torch.tensor(Hc.imag)], 1).float().to(device)
    args = SimpleNamespace(size='tinypro', patch_size=4, t_patch_size=4, no_qkv_bias=0,
        pos_emb='SinCos_3D', MoE=False, csi_enhance=1, rgb_pos_enhance=1,
        fastdepth_weights=str(ROOT / 'weights/FastDepthV2_L1_Best.pth'), device_id='0')
    m = WiFo2_model(args=args).to(device)
    m.load_state_dict(torch.load(ROOT / 'weights/model_best.pkl', map_location=device), strict=False)
    freeze_wifo(args, m, task_id=1)
    m.eval()
    with torch.no_grad():
        lat = m.forward_encoder(x, 0.0, 'fre', scale=[1, 1, 1])[0]
    return lat.mean(1).numpy()          # mean-pool the 384 tokens -> (N, 64)

X_vit, X_vit_te = wifo_features(H), wifo_features(H_te)
print('wifo train', X_vit.shape, ' test', X_vit_te.shape)

## 4. Evaluation harness

`SelectKBest` sits **inside** the pipeline, so feature selection is refit on
each training fold only. Selecting features on all 10 samples first inflates
the score by about 0.08 — the leaked version reported 0.890 where the honest
one gives 0.811.

Both 5-fold (seed-averaged) and leave-one-out are reported. LOOCV uses 9 of 10
samples per fit, so it is the more optimistic of the two.

In [ ]:
def pipe(k):
    return make_pipeline(StandardScaler(), SelectKBest(f_classif, k=k),
                         LogisticRegression(C=1, max_iter=2000))

def evaluate(X, yy, k):
    k = min(k, X.shape[1])
    scores = []
    for s in SEEDS:
        oof = np.zeros(len(yy), int)
        for itr, iva in StratifiedKFold(5, shuffle=True, random_state=s).split(X, yy):
            p = pipe(k); p.fit(X[itr], yy[itr]); oof[iva] = p.predict(X[iva])
        scores.append(f1_score(yy, oof, average='macro'))
    oof = np.zeros(len(yy), int)
    for itr, iva in LeaveOneOut().split(X):
        p = pipe(k); p.fit(X[itr], yy[itr]); oof[iva] = p.predict(X[iva])
    return np.mean(scores), np.std(scores), f1_score(yy, oof, average='macro'), oof

X_both, X_both_te = np.hstack([X_phys, X_vit]), np.hstack([X_phys_te, X_vit_te])

candidates = [
    ('physics',        X_phys, 1), ('physics',        X_phys, 2), ('physics',       X_phys, 3),
    ('wifo mean-pool', X_vit,  3), ('wifo mean-pool', X_vit,  5),
    ('physics + wifo', X_both, 2), ('physics + wifo', X_both, 3), ('physics + wifo', X_both, 5),
]

print(f"{'feature set':24s}{'dims':>6s}{'k':>4s}{'5-fold':>9s}{'std':>7s}{'LOOCV':>8s}")
results, best = {}, (0, None)
for name, X, k in candidates:
    mean, std, lo, _ = evaluate(X, y, k)
    results[f'{name} (k={k})'] = mean
    print(f'{name:24s}{X.shape[1]:>6}{k:>4}{mean:>9.3f}{std:>7.3f}{lo:>8.3f}')
    if mean > best[0]:
        best = (mean, (name, X, k, X_both_te if name == 'physics + wifo' else
                                   X_phys_te if name == 'physics' else X_vit_te))

print(f'\nbest: {best[1][0]} k={best[1][2]}  ->  {best[0]:.3f}')
print('baselines: wifo linear probe 0.428 | mean-pool probe 0.561 | majority 0.375')

## 5. Where the errors are

In [ ]:
name, Xb, k, Xb_te = best[1]
mean, std, lo, oof = evaluate(Xb, y, k)

print(f'{name} k={k}   out-of-fold (LOOCV) predictions\n')
print(f"{'sample':>7}{'true':>7}{'pred':>7}{'ok':>5}")
for i in range(len(y)):
    print(f'{i:>7}{y[i]:>7}{oof[i]:>7}{"y" if oof[i]==y[i] else "n":>5}')
print(f'\nmacro F1 {lo:.3f}   accuracy {(oof==y).mean():.2f}')
print(f'confusion (rows=true, cols=pred):\n{confusion_matrix(y, oof)}')

## 6. Which features survive fold-internal selection

A feature picked in every fold is a real signal. One picked in three folds is
the selector chasing noise.

In [ ]:
picked = {}
for itr, _ in LeaveOneOut().split(X_both):
    p = pipe(k); p.fit(X_both[itr], y[itr])
    all_names = names + [f'wifo_{i}' for i in range(X_vit.shape[1])]
    for n in np.array(all_names)[p[1].get_support()]:
        picked[n] = picked.get(n, 0) + 1

for n, c in sorted(picked.items(), key=lambda kv: -kv[1]):
    print(f'  {n:18s} {c}/10 folds')

## 7. Final model and test predictions

Refit on all 10 samples with the winning configuration, then predict the 20
unlabelled test samples.

In [ ]:
final = pipe(k)
final.fit(Xb, y)
pred_te = final.predict(Xb_te).astype(int)
prob_te = final.predict_proba(Xb_te)

print(f"{'sample':>7}{'pred':>7}{'P(NLoS)':>10}{'P(LoS)':>9}")
for i in range(len(pred_te)):
    print(f'{i:>7}{pred_te[i]:>7}{prob_te[i,0]:>10.3f}{prob_te[i,1]:>9.3f}')

print(f'\npredicted balance {np.bincount(pred_te, minlength=2)}  (train was {np.bincount(y)})')
print(f'mean confidence {prob_te.max(1).mean():.3f}')

In [ ]:
out = ROOT / 'experiments/task1_physics'
out.mkdir(parents=True, exist_ok=True)

(out / 'submission.json').write_text(json.dumps({
    'task1': pred_te.tolist(), 'task2': [], 'task3': [],
}))
(out / 'cv_results.json').write_text(json.dumps({
    'best': f'{name} k={k}', 'cv5_mean': float(mean), 'cv5_std': float(std),
    'loocv': float(lo), 'all_variants': {a: float(b) for a, b in results.items()},
    'seeds': SEEDS, 'n_train': int(len(y)),
}, indent=2))

print(f"task1 -> {pred_te.tolist()}")
print('\nRun make_submission.py to attach the task2 persistence baseline.')

## Summary

| approach | macro F1 (5-fold, 10 seeds) |
|---|---|
| majority class | 0.375 |
| WiFo2 linear probe — the repo baseline | 0.428 |
| WiFo2 mean-pool probe | 0.561 |
| WiFo2 mean-pool + logreg | 0.748 |
| physics features alone | 0.811 |
| **physics + WiFo2, k=3** | **0.880** |

Two independent gains stacked here. Replacing the SGD-trained linear head with
`StandardScaler + SelectKBest + LogisticRegression` took the WiFo2 features from
0.561 to 0.748 — the frozen features were better than the baseline head could
exploit. Adding physics features took it to 0.880.

Caveat that does not go away: 10 labelled samples. The seed spread is 0.03-0.12,
LOOCV and 5-fold disagree by up to 0.08, and none of this has been checked
against a held-out set. Treat the ordering as reliable and the absolute numbers
as optimistic.

Next candidates, in order of expected value:

- **augmentation** — global phase rotation, AWGN at known SNR, antenna permutation. All label-preserving; turns 10 samples into thousands and would let the transformer actually train
- **more physics** — per-antenna K-factor spread, Doppler from the time axis, angular spread via a spatial FFT
- **MAE pretraining on the 20 unlabelled test samples** — self-supervised, no labels needed, adapts the backbone to this deployment